In [1]:
import requests
import pandas as pd
import numpy as np

In [2]:
# ================================
# 1. CONFIGURATION
# ================================

API_KEY = "your_key"     # <-- Insert your key
ZONE = "PL"
GRANULARITY = "5_minutes"

url = (
    f"https://api.electricitymaps.com/v3/electricity-mix/latest"
    f"?zone={ZONE}"
    f"&temporalGranularity={GRANULARITY}"
    f"&flowTraced=false"
)

headers = {
    "auth-token": API_KEY
}

In [3]:
# ================================
# 2. REQUEST
# ================================

response = requests.get(url, headers=headers)

data = response.json()

print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("JSON received. Keys:", list(data.keys()))
else:
    print("Request failed.")

Status code: 200
JSON received. Keys: ['zone', 'temporalGranularity', 'unit', 'data']


In [4]:
# ================================
# 3. PARSE JSON → DATAFRAME  (fixed)
# ================================

record = data["data"][0]
mix_data = record["mix"]

df = pd.DataFrame.from_dict(mix_data, orient="index", columns=["value_mw"])

df = df.reset_index().rename(columns={"index": "source"})

df["datetime"] = record["datetime"]
df["is_estimated"] = record["isEstimated"]

df["datetime"] = pd.to_datetime(df["datetime"])

df = df.set_index("datetime")

df.head()

,source,value_mw,is_estimated
datetime,,,
2025-11-24 09:00:00+00:00,nuclear,NaN,True
2025-11-24 09:00:00+00:00,geothermal,NaN,True
2025-11-24 09:00:00+00:00,biomass,364.0,True
2025-11-24 09:00:00+00:00,coal,13784.0,True
2025-11-24 09:00:00+00:00,wind,2349.0,True


In [ ]:
df.shape

(12, 3)

In [5]:
df.isna().sum()

,0
source,0
value_mw,3
is_estimated,0


In [6]:
# ================================
# 4. CLEANING (null → 0)
# ================================

df["value_mw"] = df["value_mw"].fillna(0)

df.head()

,source,value_mw,is_estimated
datetime,,,
2025-11-24 09:00:00+00:00,nuclear,0.0,True
2025-11-24 09:00:00+00:00,geothermal,0.0,True
2025-11-24 09:00:00+00:00,biomass,364.0,True
2025-11-24 09:00:00+00:00,coal,13784.0,True
2025-11-24 09:00:00+00:00,wind,2349.0,True


In [7]:
df.isna().sum()

,0
source,0
value_mw,0
is_estimated,0


In [8]:
df.dtypes

,0
source,object
value_mw,float64
is_estimated,bool


In [ ]:
df.describe()

,value_mw
count,12.000000
mean,2066.583333
std,3885.672107
min,0.000000
25%,140.500000
50%,513.000000
75%,1608.250000
max,13413.000000


In [11]:
def get_schema(df):
    return (df.columns.tolist(), df.dtypes.tolist())

schema1 = get_schema(df)

# simulate second run (or re-run your ETL code)
schema2 = get_schema(df)

print("Schemas identical:", schema1 == schema2)

Schemas identical: True


In [10]:
# ================================
# 5. SAVE TO PARQUET
# ================================

import os

file_path = "electricity.parquet"
df.to_parquet(file_path)

if os.path.exists(file_path):
    print(f"File saved successfully: {file_path}")

File saved successfully: electricity.parquet
